<a href="https://colab.research.google.com/github/ahmedyomiisiaka/RP-Network-Analysis/blob/main/notebooks/Notebook_02_Apoptosis_Gene_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02 — Apoptosis Pathway Gene Collection

**Project:** Network-Based Analysis of Signalling Pathways in Retinitis Pigmentosa
**Institution:** Vilnius University, Systems Biology
**Supervisor:** Doc. Dr. Erinija Pranckevičienė
**Author:** Ahmed Yomi Isiaka
**Stage:** 2 — Apoptosis Pathway Gene Collection

---

## Objective

Collect the complete set of human proteins involved in the apoptosis
pathway from two independent curated sources:

1. **Reactome** — pathway R-HSA-109581 (Intrinsic Pathway for Apoptosis)
2. **KEGG** — pathway hsa04210 (Apoptosis)

These proteins form the **target nodes** for source-to-target pathway
reconstruction in Stage 6. The 112 RP protein-coding genes collected
in Stage 1 are the source nodes. PathLinker will find the shortest
paths connecting RP proteins to apoptosis proteins through the
filtered PPI network.

## Why these two sources?

Reactome and KEGG are the two most widely used curated pathway
databases in systems biology. Using both ensures complete coverage
and allows cross-validation. Proteins confirmed in both databases
carry the highest confidence for pathway membership.

## Why the intrinsic pathway?

The intrinsic apoptosis pathway is triggered by intracellular stress
signals including DNA damage, oxidative stress, calcium dysregulation,
and ER stress — all of which are documented mechanisms in photoreceptor
degeneration in RP. This makes R-HSA-109581 the most biologically
relevant apoptosis pathway for this project.

## Input
- Reactome REST API (programmatic query)
- KEGG REST API (programmatic query)

## Output

| File | Description |
|------|-------------|
| `data/raw/apoptosis_reactome.csv` | Proteins from Reactome R-HSA-109581 |
| `data/raw/apoptosis_kegg.csv` | Proteins from KEGG hsa04210 |
| `data/processed/apoptosis_genes_final.csv` | Final reconciled target list |

## References
- Gillespie M., et al. (2022). The reactome pathway knowledgebase 2022.
  *Nucleic Acids Res*, 50(D1):D687-D692.
  https://doi.org/10.1093/nar/gkab1028
- Kanehisa M., et al. (2023). KEGG for taxonomy-based analysis of
  pathways and genomes. *Nucleic Acids Res*, 51(D1):D587-D592.
  https://doi.org/10.1093/nar/gkac963

In [1]:
# ─────────────────────────────────────────────────────────────────
# Notebook 02 — Cell 1: Install libraries and create folders
# ─────────────────────────────────────────────────────────────────

import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "requests", "pandas"],
    check=True
)

import os
from pathlib import Path

DIRS = [
    "/content/rp_network/data/raw",
    "/content/rp_network/data/processed",
    "/content/rp_network/data/results",
    "/content/rp_network/docs",
    "/content/rp_network/figures",
]
for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✓ Libraries installed.")
print("✓ Folder structure ready.")

✓ Libraries installed.
✓ Folder structure ready.


In [2]:
# ─────────────────────────────────────────────────────────────────
# Notebook 02 — Cell 2: Imports
# ─────────────────────────────────────────────────────────────────

import requests
import pandas as pd
import json
import time
from datetime import datetime

print(f"pandas   : {pd.__version__}")
print(f"requests : {requests.__version__}")
print("✓ All imports successful.")

pandas   : 2.2.2
requests : 2.32.4
✓ All imports successful.


---
## Part A — Reactome Pathway R-HSA-109581

**Pathway:** Intrinsic Pathway for Apoptosis
**Reactome ID:** R-HSA-109581
**URL:** https://reactome.org/PathwayBrowser/#/R-HSA-109581

The Reactome REST API is queried using the participants endpoint
which returns all physical entities in the pathway including
proteins, complexes, small molecules, and RNA.

We filter for:
- Human proteins only (Homo sapiens)
- Entries with a valid UniProt accession
- schemaClass containing "Protein" or "ReferenceGeneProduct"

**API endpoint used:**
```
GET https://reactome.org/ContentService/data/participants/{pathwayId}
```

In [3]:
# ─────────────────────────────────────────────────────────────────
# Notebook 02 — Cell 3: Query Reactome R-HSA-109581
#
# Source   : Reactome (https://reactome.org)
# Pathway  : R-HSA-109581 — Intrinsic Pathway for Apoptosis
# Endpoint : /ContentService/data/participants/{pathwayId}
#
# The endpoint returns all physical entities in the pathway.
# We filter for UniProt-annotated human proteins only.
#
# Output   : data/raw/apoptosis_reactome.csv
# ─────────────────────────────────────────────────────────────────

REACTOME_PATHWAY_ID = "R-HSA-109581"
REACTOME_URL = (
    f"https://reactome.org/ContentService/data/participants"
    f"/{REACTOME_PATHWAY_ID}"
)

print(f"Querying Reactome API...")
print(f"Pathway  : {REACTOME_PATHWAY_ID}")
print(f"URL      : {REACTOME_URL}")
print()

response = requests.get(REACTOME_URL, timeout=60)
print(f"Status   : {response.status_code}")

if response.status_code == 200:
    data = response.json()
    print(f"Total entities returned : {len(data)}")
    print()
    print("First entry structure:")
    print(json.dumps(data[0], indent=2))
else:
    print(f"✗ Error: {response.text[:300]}")

Querying Reactome API...
Pathway  : R-HSA-109581
URL      : https://reactome.org/ContentService/data/participants/R-HSA-109581

Status   : 200
Total entities returned : 295

First entry structure:
{
  "peDbId": 140976,
  "displayName": "TRADD:TRAF2:RIP1:FADD:CASP8(1-479) [cytosol]",
  "schemaClass": "Complex",
  "refEntities": [
    {
      "dbId": 66371,
      "stId": "uniprot:Q12933",
      "identifier": "Q12933",
      "schemaClass": "ReferenceGeneProduct",
      "displayName": "UniProt:Q12933 TRAF2",
      "icon": "EntityWithAccessionedSequence",
      "url": "http://purl.uniprot.org/uniprot/Q12933"
    },
    {
      "dbId": 66377,
      "stId": "uniprot:Q15628",
      "identifier": "Q15628",
      "schemaClass": "ReferenceGeneProduct",
      "displayName": "UniProt:Q15628 TRADD",
      "icon": "EntityWithAccessionedSequence",
      "url": "http://purl.uniprot.org/uniprot/Q15628"
    },
    {
      "dbId": 63119,
      "stId": "uniprot:Q13546",
      "identifier": "Q13546",
      

In [5]:
# ─────────────────────────────────────────────────────────────────
# Notebook 02 — Cell 4 (corrected): Parse Reactome response
#
# The API response structure confirmed from Cell 3:
#   - Each entity has a "refEntities" list
#   - Each refEntity has:
#       schemaClass : "ReferenceGeneProduct" (these are proteins)
#       identifier  : UniProt accession (e.g. "Q14790")
#       displayName : "UniProt:Q14790 CASP8"
#
# There is NO speciesName field in refEntities.
# Human proteins are identified by:
#   - schemaClass == "ReferenceGeneProduct"
#   - stId starts with "uniprot:" (confirms UniProt entry)
#   - identifier is present
#
# Gene symbol is extracted from displayName:
#   "UniProt:Q14790 CASP8" → split on space → take last word
# ─────────────────────────────────────────────────────────────────

reactome_proteins = []

for entity in data:
    ref_entities = entity.get("refEntities", [])

    for ref in ref_entities:
        schema     = ref.get("schemaClass", "")
        identifier = ref.get("identifier", "")
        st_id      = ref.get("stId", "")
        display    = ref.get("displayName", "")

        # Keep only ReferenceGeneProduct with UniProt accession
        if schema != "ReferenceGeneProduct":
            continue
        if not identifier:
            continue
        if not st_id.startswith("uniprot:"):
            continue

        # Extract gene symbol from displayName
        # Format: "UniProt:Q14790 CASP8" → gene symbol = "CASP8"
        parts       = display.strip().split(" ")
        gene_symbol = parts[-1] if len(parts) > 1 else display

        reactome_proteins.append({
            "gene_symbol"  : gene_symbol,
            "uniprot_id"   : identifier,
            "protein_name" : display,
            "source"       : "Reactome",
            "pathway_id"   : REACTOME_PATHWAY_ID,
            "pathway_name" : "Intrinsic Pathway for Apoptosis",
            "date_accessed": datetime.now().strftime("%Y-%m-%d"),
        })

# Build DataFrame and remove duplicates by UniProt ID
df_reactome = pd.DataFrame(reactome_proteins)
df_reactome = df_reactome.drop_duplicates(
    subset=["uniprot_id"]
).reset_index(drop=True)

print(f"Human proteins extracted : {len(df_reactome)}")
print()
if len(df_reactome) > 0:
    print("First 15 proteins:")
    print(df_reactome[["gene_symbol","uniprot_id"]].head(15).to_string())
    print()

    # Save
    out_path = "/content/rp_network/data/raw/apoptosis_reactome.csv"
    df_reactome.to_csv(out_path, index=False)
    print(f"✓ Saved → {out_path}")
else:
    print("Still empty — check the refEntities structure above.")
    # Debug: show all schemaClass values present
    all_schemas = set()
    all_stids   = set()
    for entity in data[:5]:
        for ref in entity.get("refEntities", []):
            all_schemas.add(ref.get("schemaClass",""))
            all_stids.add(ref.get("stId","")[:10])
    print(f"SchemaClass values seen : {all_schemas}")
    print(f"stId prefixes seen      : {all_stids}")

Human proteins extracted : 165

First 15 proteins:
   gene_symbol uniprot_id
0        TRAF2     Q12933
1        TRADD     Q15628
2        RIPK1     Q13546
3         FADD     Q13158
4        CASP8     Q14790
5    TNFRSF10B     O14763
6    TNFRSF10A     O00220
7      TNFSF10     P50591
8        FASLG     P48023
9          FAS     P25445
10        CD14     P08571
11        TLR4     O00206
12        LY96     Q9Y6Y9
13      TICAM2     Q86XR7
14      TICAM1     Q8IUC6

✓ Saved → /content/rp_network/data/raw/apoptosis_reactome.csv


---
## Part B — KEGG Pathway hsa04210

**Pathway:** Apoptosis
**KEGG ID:** hsa04210
**URL:** https://www.genome.jp/pathway/hsa04210

KEGG's apoptosis pathway covers both intrinsic and extrinsic
routes and is broader than the Reactome intrinsic pathway.
Using both ensures complete coverage.

**API steps:**

Step 1 — Get all gene IDs in the pathway:
GET https://rest.kegg.jp/link/hsa/hsa04210


Step 2 — Get gene names for each ID:

GET https://rest.kegg.jp/list/{gene_ids}


Step 3 — Get UniProt accessions:

GET https://rest.kegg.jp/conv/uniprot/hsa04210

In [6]:
# ─────────────────────────────────────────────────────────────────
# Notebook 02 — Cell 5: Query KEGG — Step 1
#
# Get all KEGG gene IDs in pathway hsa04210
#
# The /link endpoint returns tab-separated pairs:
#   pathway:hsa04210  hsa:ENTREZ_ID
# ─────────────────────────────────────────────────────────────────

KEGG_PATHWAY = "hsa04210"

print(f"Step 1: Getting gene IDs from KEGG pathway {KEGG_PATHWAY}...")
link_url = f"https://rest.kegg.jp/link/hsa/{KEGG_PATHWAY}"
r = requests.get(link_url, timeout=60)
print(f"Status : {r.status_code}")

if r.status_code != 200:
    print(f"Error  : {r.text[:200]}")
else:
    kegg_ids = []
    for line in r.text.strip().splitlines():
        parts = line.split("\t")
        if len(parts) == 2:
            kegg_id = parts[1].strip()
            kegg_ids.append(kegg_id)

    kegg_ids = list(set(kegg_ids))
    print(f"Unique KEGG gene IDs : {len(kegg_ids)}")
    print(f"Sample               : {kegg_ids[:5]}")

Step 1: Getting gene IDs from KEGG pathway hsa04210...
Status : 200
Unique KEGG gene IDs : 138
Sample               : ['hsa:5293', 'hsa:10376', 'hsa:841', 'hsa:3551', 'hsa:8772']


In [7]:
# ─────────────────────────────────────────────────────────────────
# Notebook 02 — Cell 6: Query KEGG — Step 2
#
# Get gene symbols and descriptions for all KEGG IDs.
# KEGG API processes up to 10 IDs per request.
# We batch the requests and pause between calls.
# ─────────────────────────────────────────────────────────────────

print(f"Step 2: Getting gene info for {len(kegg_ids)} IDs...")
print()

kegg_gene_info = {}
batch_size = 10

for i in range(0, len(kegg_ids), batch_size):
    batch = kegg_ids[i:i+batch_size]
    url   = f"https://rest.kegg.jp/list/{'+'.join(batch)}"
    r     = requests.get(url, timeout=60)

    if r.status_code == 200:
        for line in r.text.strip().splitlines():
            parts = line.split("\t")
            if len(parts) >= 2:
                kid      = parts[0].strip()
                desc     = parts[1].strip()
                gene_sym = desc.split(";")[0].split(",")[0].strip()
                kegg_gene_info[kid] = {
                    "gene_symbol": gene_sym,
                    "description": desc,
                }
    time.sleep(0.3)

print(f"Gene info retrieved : {len(kegg_gene_info)} IDs")
print()
print("Sample entries:")
for k, v in list(kegg_gene_info.items())[:5]:
    print(f"  {k} → {v['gene_symbol']} | {v['description'][:60]}")

Step 2: Getting gene info for 138 IDs...

Gene info retrieved : 138 IDs

Sample entries:
  hsa:5293 → PIK3CD | PIK3CD, APDS, IMD14, IMD14A, IMD14B, P110DELTA, PI3K, ROCHIS
  hsa:10376 → TUBA1B | TUBA1B, K-ALPHA-1; tubulin alpha-1B chain
  hsa:841 → CASP8 | CASP8, ALPS2B, CAP4, Casp-8, FLICE, MACH, MCH5; caspase-8 is
  hsa:3551 → IKBKB | IKBKB, IKK-2, IKK-beta, IKK2, IKKB, IMD15, IMD15A, IMD15B, N
  hsa:8772 → FADD | FADD, GIG3, IMD90, MORT1; FAS-associated death domain protei


In [10]:
# ─────────────────────────────────────────────────────────────────
# Notebook 02 — Cell 7 (corrected): Convert KEGG IDs to UniProt
#
# The endpoint conv/uniprot/hsa04210 does not work for pathway IDs.
# Correct approach:
#   Use https://rest.kegg.jp/conv/uniprot/hsa
#   This returns ALL human KEGG → UniProt mappings.
#   We then filter for only the IDs in our pathway.
#
# Alternative for any missing IDs:
#   Use https://rest.kegg.jp/get/{kegg_id}
#   and parse the UniProt accession from the gene entry.
# ─────────────────────────────────────────────────────────────────

print("Step 3: Getting KEGG → UniProt mapping for all human genes...")
print("(This downloads the full human mapping table and filters it)")
print()

conv_url = "https://rest.kegg.jp/conv/uniprot/hsa"
r_conv   = requests.get(conv_url, timeout=120)
print(f"Status : {r_conv.status_code}")

kegg_to_uniprot = {}

if r_conv.status_code == 200:
    for line in r_conv.text.strip().splitlines():
        parts = line.split("\t")
        if len(parts) == 2:
            kid     = parts[0].strip()   # hsa:207
            uniprot = parts[1].replace("up:", "").strip()
            kegg_to_uniprot[kid] = uniprot

    print(f"Total human KEGG→UniProt mappings : {len(kegg_to_uniprot)}")

    # Filter for our pathway genes only
    pathway_mappings = {
        kid: uniprot
        for kid, uniprot in kegg_to_uniprot.items()
        if kid in kegg_ids
    }
    print(f"Mappings for hsa04210 genes       : {len(pathway_mappings)}")
    print()
    print("Sample mappings:")
    for k, v in list(pathway_mappings.items())[:5]:
        gene = kegg_gene_info.get(k, {}).get("gene_symbol", "?")
        print(f"  {k} ({gene}) → {v}")

    # Update kegg_to_uniprot to only contain pathway genes
    kegg_to_uniprot = pathway_mappings

else:
    print(f"Error: {r_conv.text[:200]}")

Step 3: Getting KEGG → UniProt mapping for all human genes...
(This downloads the full human mapping table and filters it)

Status : 200
Total human KEGG→UniProt mappings : 19417
Mappings for hsa04210 genes       : 138

Sample mappings:
  hsa:10000 (AKT3) → Q9Y243
  hsa:10018 (BCL2L11) → O43521
  hsa:10038 (PARP2) → Q9UGN5
  hsa:10039 (PARP3) → Q9Y6F1
  hsa:100506742 (CASP12) → Q6UXS9


In [11]:
# ─────────────────────────────────────────────────────────────────
# Notebook 02 — Cell 8 (corrected): Build KEGG apoptosis table
#
# Combines gene symbols, descriptions, and UniProt IDs.
# Genes without a UniProt mapping are reported separately.
#
# Output: data/raw/apoptosis_kegg.csv
# ─────────────────────────────────────────────────────────────────

kegg_rows    = []
no_uniprot   = []

for kid in kegg_ids:
    gene_info = kegg_gene_info.get(kid, {})
    uniprot   = kegg_to_uniprot.get(kid, "")
    gene_sym  = gene_info.get("gene_symbol", "")

    if not gene_sym:
        continue

    if uniprot:
        kegg_rows.append({
            "gene_symbol"  : gene_sym,
            "kegg_id"      : kid,
            "uniprot_id"   : uniprot,
            "description"  : gene_info.get("description", ""),
            "source"       : "KEGG",
            "pathway_id"   : KEGG_PATHWAY,
            "pathway_name" : "Apoptosis",
            "date_accessed": datetime.now().strftime("%Y-%m-%d"),
        })
    else:
        no_uniprot.append(gene_sym)

df_kegg = pd.DataFrame(kegg_rows)
df_kegg = df_kegg.drop_duplicates(
    subset=["uniprot_id"]
).reset_index(drop=True)

print(f"KEGG proteins with UniProt IDs    : {len(df_kegg)}")
print(f"KEGG proteins WITHOUT UniProt IDs : {len(no_uniprot)}")
if no_uniprot:
    print(f"  Genes without mapping: {no_uniprot}")
print()
print("First 15 proteins:")
print(df_kegg[["gene_symbol","kegg_id","uniprot_id"]].head(15).to_string())

# Save
out_path = "/content/rp_network/data/raw/apoptosis_kegg.csv"
df_kegg.to_csv(out_path, index=False)
print()
print(f"✓ Saved → {out_path}")

KEGG proteins with UniProt IDs    : 137
KEGG proteins WITHOUT UniProt IDs : 0

First 15 proteins:
   gene_symbol    kegg_id  uniprot_id
0       PIK3CD   hsa:5293  A0A2K8FKV1
1       TUBA1B  hsa:10376      P68363
2        CASP8    hsa:841      Q14790
3        IKBKB   hsa:3551      O14920
4         FADD   hsa:8772      Q13158
5      GADD45A   hsa:1647      P24522
6        RIPK1   hsa:8737      Q13546
7        TUBA8  hsa:51807      Q9NY65
8      BCL2L11  hsa:10018      O43521
9      EIF2AK3   hsa:9451      B3KY45
10     TNFSF10   hsa:8743      Q6IBA9
11        RELA   hsa:5970      Q04206
12        BAK1    hsa:578  A0A0S2Z391
13        CHUK   hsa:1147      O15111
14        CTSZ   hsa:1522      Q9UBR2

✓ Saved → /content/rp_network/data/raw/apoptosis_kegg.csv


---
## Part C — Reconciliation

We compare both lists to identify:
1. Proteins confirmed in **both** sources — highest confidence targets
2. Proteins **only in Reactome** — intrinsic pathway specific
3. Proteins **only in KEGG** — broader apoptosis coverage

All proteins from both sources are included in the final list
with a confidence annotation. This gives PathLinker the most
complete set of apoptosis target nodes.

In [12]:
# ─────────────────────────────────────────────────────────────────
# Notebook 02 — Cell 9: Cross-check Reactome and KEGG
#
# Comparison is performed on gene symbols (uppercase).
# UniProt IDs are also checked for additional overlap detection.
# ─────────────────────────────────────────────────────────────────

reactome_symbols = set(df_reactome["gene_symbol"].str.upper().dropna())
kegg_symbols     = set(df_kegg["gene_symbol"].str.upper().dropna())

in_both       = reactome_symbols & kegg_symbols
only_reactome = reactome_symbols - kegg_symbols
only_kegg     = kegg_symbols     - reactome_symbols

print("=" * 55)
print("CROSS-CHECK RESULTS")
print("=" * 55)
print(f"Reactome R-HSA-109581 proteins : {len(reactome_symbols)}")
print(f"KEGG hsa04210 proteins         : {len(kegg_symbols)}")
print()
print(f"In BOTH sources                : {len(in_both)}")
print(f"Only in Reactome               : {len(only_reactome)}")
print(f"Only in KEGG                   : {len(only_kegg)}")
print()
print("Proteins in BOTH (core apoptosis targets):")
for g in sorted(in_both):
    print(f"  {g}")

CROSS-CHECK RESULTS
Reactome R-HSA-109581 proteins : 165
KEGG hsa04210 proteins         : 137

In BOTH sources                : 40
Only in Reactome               : 125
Only in KEGG                   : 97

Proteins in BOTH (core apoptosis targets):
  AKT1
  AKT2
  AKT3
  APAF1
  BAD
  BAK1
  BAX
  BBC3
  BCL2
  BCL2L1
  BCL2L11
  BID
  BIRC2
  CASP3
  CASP6
  CASP7
  CASP8
  CASP9
  CYCS
  DFFA
  DFFB
  DIABLO
  FADD
  FAS
  FASLG
  GZMB
  LMNB1
  MAPK1
  MAPK3
  MAPK8
  PMAIP1
  RIPK1
  SPTAN1
  TNFRSF10A
  TNFRSF10B
  TNFSF10
  TP53
  TRADD
  TRAF2
  XIAP


In [13]:
# ─────────────────────────────────────────────────────────────────
# Notebook 02 — Cell 10: Build final apoptosis target list
#
# All proteins from both sources are included.
# Each entry is annotated with:
#   gene_symbol    : HGNC gene symbol
#   uniprot_id     : UniProt accession
#   in_reactome    : True/False
#   in_kegg        : True/False
#   confidence     : Both sources / Reactome only / KEGG only
#   source_primary : which database listed it
#   pathway_ids    : which pathways it belongs to
#   date_compiled  : today
#
# Output: data/processed/apoptosis_genes_final.csv
# ─────────────────────────────────────────────────────────────────

final_rows = []

# --- Reactome proteins ---
for _, row in df_reactome.iterrows():
    sym    = str(row["gene_symbol"]).upper()
    in_k   = sym in kegg_symbols

    final_rows.append({
        "gene_symbol"  : row["gene_symbol"],
        "uniprot_id"   : row["uniprot_id"],
        "protein_name" : row["protein_name"],
        "in_reactome"  : True,
        "in_kegg"      : in_k,
        "confidence"   : "Both sources" if in_k else "Reactome only",
        "pathway_ids"  : "R-HSA-109581" if not in_k
                         else "R-HSA-109581, hsa04210",
        "source_primary": "Reactome",
        "date_compiled": datetime.now().strftime("%Y-%m-%d"),
    })

# --- KEGG-only proteins ---
for _, row in df_kegg.iterrows():
    sym = str(row["gene_symbol"]).upper()
    if sym not in reactome_symbols:
        final_rows.append({
            "gene_symbol"  : row["gene_symbol"],
            "uniprot_id"   : row["uniprot_id"],
            "protein_name" : row["description"],
            "in_reactome"  : False,
            "in_kegg"      : True,
            "confidence"   : "KEGG only",
            "pathway_ids"  : "hsa04210",
            "source_primary": "KEGG",
            "date_compiled": datetime.now().strftime("%Y-%m-%d"),
        })

df_apoptosis = pd.DataFrame(final_rows)

# Remove duplicates by UniProt ID
df_apoptosis = df_apoptosis.drop_duplicates(
    subset=["uniprot_id"]
).reset_index(drop=True)

print("=" * 55)
print("FINAL APOPTOSIS TARGET LIST SUMMARY")
print("=" * 55)
print(f"Total proteins           : {len(df_apoptosis)}")
print()
print("Confidence breakdown:")
print(df_apoptosis["confidence"].value_counts().to_string())
print()
print("Source breakdown:")
print(df_apoptosis["source_primary"].value_counts().to_string())

# Save
out_path = "/content/rp_network/data/processed/apoptosis_genes_final.csv"
df_apoptosis.to_csv(out_path, index=False)
print()
print(f"✓ Saved → {out_path}")

FINAL APOPTOSIS TARGET LIST SUMMARY
Total proteins           : 262

Confidence breakdown:
confidence
Reactome only    125
KEGG only         97
Both sources      40

Source breakdown:
source_primary
Reactome    165
KEGG         97

✓ Saved → /content/rp_network/data/processed/apoptosis_genes_final.csv


In [14]:
# ─────────────────────────────────────────────────────────────────
# Notebook 02 — QC Cell: Quality control checks
#
# Verify the final apoptosis dataset before closing Stage 2.
# ─────────────────────────────────────────────────────────────────

df_check = pd.read_csv(
    "/content/rp_network/data/processed/apoptosis_genes_final.csv"
)

print("=" * 55)
print("QC REPORT — apoptosis_genes_final.csv")
print("=" * 55)

print(f"1. Total entries          : {len(df_check)}")

n_dup = df_check["uniprot_id"].duplicated().sum()
print(f"2. Duplicate UniProt IDs  : {n_dup}",
      "✓" if n_dup == 0 else "✗ PROBLEM")

n_empty_sym = df_check["gene_symbol"].isna().sum()
print(f"3. Empty gene symbols     : {n_empty_sym}",
      "✓" if n_empty_sym == 0 else "✗ PROBLEM")

n_empty_uni = df_check["uniprot_id"].isna().sum()
print(f"4. Empty UniProt IDs      : {n_empty_uni}",
      "✓" if n_empty_uni == 0 else "✗ PROBLEM")

print(f"\n5. Confidence breakdown:")
print(df_check["confidence"].value_counts().to_string())

# Check key apoptosis proteins are present
key_proteins = ["CASP3","CASP9","CASP8","BCL2","BAX","CYCS",
                "APAF1","TP53","BID","MCL1","XIAP","PARP1"]
present = [p for p in key_proteins
           if p in df_check["gene_symbol"].str.upper().values]
missing = [p for p in key_proteins
           if p not in df_check["gene_symbol"].str.upper().values]

print(f"\n6. Key apoptosis proteins present : {len(present)}/{len(key_proteins)}")
print(f"   Present : {present}")
if missing:
    print(f"   Missing : {missing}")

print()
print("=" * 55)
print("QC COMPLETE")
print("=" * 55)

QC REPORT — apoptosis_genes_final.csv
1. Total entries          : 262
2. Duplicate UniProt IDs  : 0 ✓
3. Empty gene symbols     : 0 ✓
4. Empty UniProt IDs      : 0 ✓

5. Confidence breakdown:
confidence
Reactome only    125
KEGG only         97
Both sources      40

6. Key apoptosis proteins present : 12/12
   Present : ['CASP3', 'CASP9', 'CASP8', 'BCL2', 'BAX', 'CYCS', 'APAF1', 'TP53', 'BID', 'MCL1', 'XIAP', 'PARP1']

QC COMPLETE


---
## Limitations

1. **Reactome R-HSA-109581** covers the intrinsic apoptosis pathway
   specifically. The extrinsic (death receptor) pathway is covered
   by KEGG hsa04210 but not fully by this Reactome pathway.
   Using both databases ensures broader coverage.

2. **KEGG API** returns ENTREZ gene IDs which must be converted to
   UniProt accessions. Some genes may lack a UniProt mapping and
   are excluded. This is documented in the QC output above.

3. **Protein isoforms** — UniProt accessions may refer to canonical
   or isoform sequences. For network analysis we retain the canonical
   accession as returned by the API.

4. **Pathway membership** does not imply direct physical interaction.
   Apoptosis pathway proteins are used as target nodes for
   PathLinker — the shortest-path algorithm will identify which
   of these targets are most connected to RP genes through the
   PPI network.

---

## Conclusion

Stage 2 is complete. The following has been established:

- Apoptosis target proteins collected from two independent sources
- Cross-checked and reconciled into one final target list
- Key proteins verified (CASP3, CASP9, BCL2, BAX, CYCS, APAF1)
- All entries carry UniProt accessions for downstream mapping

The final apoptosis target list will be used as the **target node
set** in Stage 6 (PathLinker pathway reconstruction).

**Next:** Notebook 03 — Gene Identifier Mapping (UniProt)

In [15]:
# ─────────────────────────────────────────────────────────────────
# Notebook 02 — Final Cell: Update progress log and save to Drive
# ─────────────────────────────────────────────────────────────────

from google.colab import drive
import shutil
drive.mount("/content/drive")

today = datetime.now().strftime("%Y-%m-%d")

log_addition = f"""
---

## Entry 3 — {today}

### Objectives for today
- Collect apoptosis pathway proteins from Reactome R-HSA-109581
- Collect apoptosis pathway proteins from KEGG hsa04210
- Cross-check both sources and produce final apoptosis target list

### Stage 2 — Apoptosis Pathway Gene Collection

#### Data sources
- Reactome pathway R-HSA-109581 (Intrinsic Pathway for Apoptosis)
- KEGG pathway hsa04210 (Apoptosis)

#### Results

| Item | Count |
|------|-------|
| Reactome R-HSA-109581 proteins | {len(df_reactome)} |
| KEGG hsa04210 proteins | {len(df_kegg)} |
| Proteins in both sources | {len(in_both)} |
| Reactome only | {len(only_reactome)} |
| KEGG only | {len(only_kegg)} |
| **Total final apoptosis targets** | **{len(df_apoptosis)}** |

#### Key proteins confirmed present
CASP3, CASP9, BCL2, BAX, CYCS, APAF1

#### Output files
- data/raw/apoptosis_reactome.csv
- data/raw/apoptosis_kegg.csv
- data/processed/apoptosis_genes_final.csv

#### Problems encountered
- Document any issues here

### Next steps (Stage 3 — Notebook 03)
- Map all RP gene symbols to UniProt accessions
- Map all apoptosis gene symbols to UniProt accessions
- Produce unified identifier table for network construction

---
"""

# Append to progress log
log_path = "/content/rp_network/docs/progress_log.md"
drive_log = "/content/drive/MyDrive/rp_network/docs/progress_log.md"

with open(drive_log, "a") as f:
    f.write(log_addition)

# Copy from Drive to Colab
shutil.copy(drive_log, log_path)

# Save all outputs to Drive
shutil.copytree(
    "/content/rp_network",
    "/content/drive/MyDrive/rp_network",
    dirs_exist_ok=True
)

print("✓ Progress log updated.")
print("✓ All files saved to Google Drive.")
print()
print("Stage 2 outputs saved:")
print("  data/raw/apoptosis_reactome.csv")
print("  data/raw/apoptosis_kegg.csv")
print("  data/processed/apoptosis_genes_final.csv")
print("  docs/progress_log.md")

Mounted at /content/drive
✓ Progress log updated.
✓ All files saved to Google Drive.

Stage 2 outputs saved:
  data/raw/apoptosis_reactome.csv
  data/raw/apoptosis_kegg.csv
  data/processed/apoptosis_genes_final.csv
  docs/progress_log.md


In [16]:
from google.colab import files
files.download("/content/rp_network/data/processed/apoptosis_genes_final.csv")
files.download("/content/rp_network/docs/progress_log.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>